# Main実験
上から実行すると共通bankを準備し、A/B/F×3 seedを2本並列で30kまで学習します。decision評価・確率校正は別途有効にします。

In [ ]:
from pathlib import Path
import json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
import sys
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from uav_safe_marl.experiments.main import evaluate_condition, context
from uav_safe_marl.experiments.main_analysis import analyze
from scripts.main_notebook_runner import probe_training_device, run_training_grid
OUTPUT = ROOT / 'results' / 'main_four_faces_free_30m'
SETTINGS = ROOT / 'notebooks' / 'main' / 'settings.yaml'
print(OUTPUT)


## 共通設定
学習セルでbankを一度だけ準備します。設定を変更する場合は新しいOUTPUTを使います。確認Testは生成のみで、評価しません。

In [ ]:
WORKERS = 2
TARGET = 30000


## 学習デバイスの確認
WindowsでGeForceを使う場合は、次のセルで `cuda:0` などのCUDAデバイス番号と、目的のGeForceの製品名を確認してから学習セルへ進みます。

In [ ]:
hardware = probe_training_device(ROOT, SETTINGS)
gpu = hardware['cuda_device_name']
device = hardware['training_device']
print(f"学習デバイス: {device} ({gpu})" if gpu else f"学習デバイス: {device}")
print(f"学習デバイス設定: {hardware['requested_device']} / 評価デバイス: {hardware['evaluation_device']}")
print(f"PyTorch: {hardware['torch']} / CUDAビルド: {hardware['torch_cuda'] or 'なし'} / CUDA利用可能: {hardware['cuda_available']}")
print(f"Python: {hardware['python']}")


## 学習と継続
このセルで全9 runを起動します。tqdmのバー1本に完了run数と、実行中の各runの累積online step・完了episode数を表示します。別のNotebookやPowerShellから同じ出力先の学習を同時に起動しないでください。30kの全development結果を見て延長を決めた場合だけ、同じOUTPUTでTARGET=60000に変更して実行します。

In [ ]:
from tqdm.notebook import tqdm

completed_runs = run_training_grid(
    ROOT, SETTINGS, OUTPUT, target=TARGET, workers=WORKERS, progress_factory=tqdm
)
print(f"学習・development完了: {len(completed_runs)} run")


## 判断用評価
A/B/C/D/F/F_OFFは各seed、E/Rは1回。

In [ ]:
RUN_EVALUATION = False
METHODS = ['A', 'B', 'C', 'D', 'E', 'F', 'R', 'F_OFF']
if RUN_EVALUATION:
    for method in METHODS:
        seeds = [0] if method in ('E', 'R') else context(OUTPUT)['settings']['training_seeds']
        for seed in seeds:
            evaluate_condition(OUTPUT, method, seed, checkpoint_step=TARGET)


## 確率方策校正
HOCBF ON・同じ初期sample actionで校正する。

In [ ]:
RUN_CALIBRATION = False
if RUN_CALIBRATION:
    for method in ['A', 'B', 'F']:
        for seed in context(OUTPUT)['settings']['training_seeds']:
            evaluate_condition(OUTPUT, method, seed, checkpoint_step=TARGET, stochastic=True)


## 保存結果の分析
途中結果も分析可能。実行済みepisode数を確認する。

In [ ]:
RUN_ANALYSIS = False
ANALYSIS_SPLIT = "development"  # decision評価後は "decision"
if RUN_ANALYSIS:
    report = analyze(OUTPUT, split=ANALYSIS_SPLIT, target=TARGET)
    from IPython.display import display, Markdown
    display(Markdown(report.read_text()))
